# 07 — Dashboard: herramienta de exploración de perfiles

**Objetivo:** construir la herramienta de apoyo a la decisión que consume los perfiles generados
en `05_clustering` (K-Means, K=5) y, opcionalmente, la representación semántica de `06_embeddings`,
para los casos de uso descritos en `context/00-contexto-permanente.md`: asignación de tareas,
conformación de comisiones/equipos, planificación académica y administrativa.

**Decisión de herramienta (ver DEC-003 en `context/DECISION_LOG.md`):** el prototipo se construye
como una aplicación **Streamlit** (`app.py`, en esta misma carpeta), no como un notebook con
widgets. Motivo: el contexto permanente del proyecto pide explícitamente un "prototipo/dashboard"
pensado desde el inicio para "eventual despliegue, mantenimiento, actualización de datos" y para
apoyar a unidades institucionales (RRHH, decanatos) que no son usuarios técnicos — un notebook no
es la interfaz adecuada para ese público ni para ese ciclo de vida. Este notebook
(`07_dashboard.ipynb`) se usa entonces para:

1. Preparar y validar los datasets consolidados que consume la aplicación (`data/dashboard/`).
2. Verificar rápidamente que los clusters no son un proxy trivial de variables demográficas
   (sexo, edad) — relevante para las restricciones conceptuales del proyecto.
3. Documentar y probar la búsqueda semántica opcional sobre `embeddings_personas.csv`.
4. Dejar instrucciones para ejecutar la aplicación.

**Entradas:**

- `data/clustering/clusters_personas.csv`, `cluster_sizes.csv`, `cluster_characterization.csv`
  (de `05_clustering`).
- `data/features/dataset_personas_features.csv`, `feature_dictionary.csv`, `personas.csv`.
- `data/embeddings/embeddings_personas.csv`, `corpus_texto_detalle.csv`,
  `cobertura_texto_personas.csv` (de `06_embeddings`).

**Salidas (en `data/dashboard/`):** `personas_dashboard.csv`, `cluster_perfiles_resumen.csv`,
`cluster_top_features.csv`.

**Nota de entorno:** este notebook usa un entorno virtual aislado (`.venv/`, kernel
*"Proyecto Tesis - Dashboard (.venv)"*) en vez del intérprete global usado por los notebooks 01-06
(kernel *"proyecto-tesis-py311"*), para poder instalar `streamlit`/`plotly` sin generar conflictos
de dependencias con otras herramientas ya instaladas globalmente (ver DEC-003).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "07_dashboard") else Path.cwd()
sys.path.insert(0, str(Path.cwd()))
import lib

DATA_FEATURES = ROOT / "data" / "features"
DATA_CLUSTERING = ROOT / "data" / "clustering"
DATA_EMBEDDINGS = ROOT / "data" / "embeddings"
DATA_DASHBOARD = ROOT / "data" / "dashboard"
DATA_DASHBOARD.mkdir(parents=True, exist_ok=True)

print("Salidas del dashboard en:", DATA_DASHBOARD)

## 1. Consolidación de datos por persona

Se une `clusters_personas.csv` (asignación de cluster) con las 85 features originales de
`dataset_personas_features.csv` y la cobertura de texto de `06_embeddings`, y se agrega el nombre
de perfil (sección 8 de `05_clustering`, sintetizado en `lib.PERFIL_NOMBRES`).

In [ ]:
clusters = pd.read_csv(DATA_CLUSTERING / "clusters_personas.csv")
features = pd.read_csv(DATA_FEATURES / "dataset_personas_features.csv")
cobertura = pd.read_csv(DATA_EMBEDDINGS / "cobertura_texto_personas.csv")

assert set(clusters["IDPERSONA"]) == set(features["IDPERSONA"]), "IDPERSONA no coincide entre clustering y features"

personas_dashboard = clusters.merge(features, on="IDPERSONA", how="left")
personas_dashboard = personas_dashboard.merge(cobertura, on="IDPERSONA", how="left")
personas_dashboard["PERFIL_NOMBRE"] = personas_dashboard["CLUSTER"].map(lib.PERFIL_NOMBRES)

cols = ["IDPERSONA", "CLUSTER", "PERFIL_NOMBRE"] + [
    c for c in personas_dashboard.columns if c not in ("IDPERSONA", "CLUSTER", "PERFIL_NOMBRE")
]
personas_dashboard = personas_dashboard[cols]

personas_dashboard.to_csv(DATA_DASHBOARD / "personas_dashboard.csv", index=False)
print(personas_dashboard.shape)
personas_dashboard.head(3)

**Nota sobre valores faltantes:** columnas como `PROMEDIO_ESTUDIANTES_POR_CURSO`,
`PROMEDIO_HETEROEVALUACION` o `DEDICACION_DOCENTE_ACTUAL` tienen NaN para personas sin actividad
docente (no aplica), no un dato perdido por error de captura — igual que en `dataset_personas_features.csv`
(ver `data/features/feature_dictionary.csv`). La aplicación debe mostrar estos casos como
"no aplica", no como "sin dato".

## 2. Resumen por cluster (para tarjetas de perfil en la app)

In [ ]:
sizes = pd.read_csv(DATA_CLUSTERING / "cluster_sizes.csv")
resumen = sizes.copy()
resumen["PERFIL_NOMBRE"] = resumen["CLUSTER"].map(lib.PERFIL_NOMBRES)
resumen["DESCRIPCION"] = resumen["CLUSTER"].map(lib.PERFIL_DESCRIPCIONES)
resumen.to_csv(DATA_DASHBOARD / "cluster_perfiles_resumen.csv", index=False)
resumen

In [ ]:
charac = pd.read_csv(DATA_CLUSTERING / "cluster_characterization.csv")
top_features = (
    charac.sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False])
    .groupby("CLUSTER")
    .head(8)
    .reset_index(drop=True)
)
top_features.to_csv(DATA_DASHBOARD / "cluster_top_features.csv", index=False)
top_features.head(10)

## 3. Verificación visual: tamaño y composición de los perfiles

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = resumen.sort_values("CLUSTER")
ax.bar(order["PERFIL_NOMBRE"], order["N_PERSONAS"], color=sns.color_palette("Set2", len(order)))
ax.set_ylabel("N personas")
ax.set_title("Tamaño de cada perfil")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
tab_tipo = pd.crosstab(personas_dashboard["PERFIL_NOMBRE"], personas_dashboard["TIPOEMPLEADO_ACTUAL_DESC"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(tab_tipo.round(1), annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% dentro del perfil"}, ax=ax)
ax.set_title("Composición por tipo de empleado (%)")
plt.tight_layout()
plt.show()
tab_tipo.round(1)

Confirma lo documentado en `05_clustering` (sección 8): el perfil **Administrativo** concentra casi
todo el personal administrativo, mientras que los otros cuatro perfiles son mayoritariamente
docentes — coherente con los nombres propuestos, sin ser una regla perfecta (algunas personas con
`ES_DOCENTE_ADMIN_MIXTO=True` aparecen en perfiles docentes).

## 4. Chequeo de equidad: ¿los perfiles son un proxy de sexo o edad?

El proyecto exige explícitamente no presentar los perfiles como verdades absolutas y evitar que
sirvan de base indirecta para decisiones sensibles (ver `context/00-contexto-permanente.md`,
restricciones conceptuales). `SEXO` y `EDAD` **no se usaron como variables de clustering** (no
están en `X_modelado.csv`); aquí se verifica, solo como chequeo de sanidad, que la distribución de
estas variables no varíe de forma extrema entre perfiles — si lo hiciera, sería una señal de alerta
sobre variables proxy dentro de las features usadas (antigüedad, tipo de cargo, etc.) que ameritaría
revisión adicional antes de usar los perfiles operativamente.

In [ ]:
demograficos = pd.read_csv(DATA_FEATURES / "personas.csv")[["IDPERSONA", "SEXO", "EDAD"]]
check = personas_dashboard[["IDPERSONA", "CLUSTER", "PERFIL_NOMBRE"]].merge(demograficos, on="IDPERSONA", how="left")

tab_sexo = pd.crosstab(check["PERFIL_NOMBRE"], check["SEXO"], normalize="index") * 100
display(tab_sexo.round(1))

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=check, x="PERFIL_NOMBRE", y="EDAD", ax=ax)
ax.set_title("Distribución de edad por perfil")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

**Lectura:** ni `SEXO` ni `EDAD` se usan como filtro ni como variable en la aplicación — se
muestran aquí únicamente como chequeo de equidad, no como parte de la ficha de persona ni de los
criterios de búsqueda de la sección "Formar equipos" de la app, precisamente para no introducir
esas variables como criterio de selección de personal.

## 5. Búsqueda semántica (opcional) — prueba de concepto

Se prueba la búsqueda semántica de texto libre sobre `embeddings_personas.csv` (384 dimensiones,
`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, ver DEC-002): se codifica la
consulta con el mismo modelo en tiempo de consulta y se calcula similitud coseno contra los vectores
ya calculados por persona (fuerza bruta con `numpy`, viable para 2196 personas × 384 dimensiones sin
necesidad de un índice aproximado). La aplicación (`app.py`) reutiliza esta misma lógica
(`lib.load_embeddings`).

In [ ]:
from sentence_transformers import SentenceTransformer

ids, X = lib.load_embeddings()
modelo_texto = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def buscar_personas(consulta, top_n=5):
    q = modelo_texto.encode([consulta], normalize_embeddings=True)[0]
    sims = X @ q
    top_idx = np.argsort(-sims)[:top_n]
    return pd.DataFrame({"IDPERSONA": ids[top_idx], "SIMILITUD": sims[top_idx]})

ejemplo = buscar_personas("experiencia en inteligencia artificial aplicada a imágenes médicas")
ejemplo = ejemplo.merge(personas_dashboard[["IDPERSONA", "CLUSTER", "PERFIL_NOMBRE"]], on="IDPERSONA", how="left")
ejemplo

**Limitaciones de la búsqueda semántica** (mostrar siempre en la app junto a los resultados):

- Solo 2196 de 2213 personas (99.2%) tienen algún texto fuente; el resto no puede aparecer en estos
  resultados aunque sea relevante (columna `N_REGISTROS_TEXTO=0` en `cobertura_texto_personas.csv`).
- La similitud semántica **no es una medida de idoneidad**: una alta similitud indica que el texto
  histórico de esa persona (publicaciones, capacitaciones, ponencias, etc.) se relaciona
  temáticamente con la consulta, no que sea la mejor opción para una tarea — requiere criterio
  humano, igual que el resto de la herramienta.

## 6. Resumen y cómo ejecutar la aplicación

**Archivos generados en `data/dashboard/`:**

| Archivo | Contenido |
|---|---|
| `personas_dashboard.csv` | Una fila por persona: `IDPERSONA`, `CLUSTER`, `PERFIL_NOMBRE` + 85 features originales + cobertura de texto |
| `cluster_perfiles_resumen.csv` | Tamaño, % de población, nombre y descripción por cluster |
| `cluster_top_features.csv` | Top 8 variables más distintivas por cluster (de `cluster_characterization.csv`) |

**Cómo ejecutar la aplicación (`app.py`, en esta misma carpeta):**

```
d:\Proyecto_Tesis\.venv\Scripts\streamlit run notebooks\07_dashboard\app.py
```

La primera vez que se use la pestaña de búsqueda semántica, la app carga el modelo de embeddings
en memoria (unos segundos); ya está cacheado localmente por `06_embeddings`, por lo que no requiere
conexión a internet.

**Limitaciones y advertencias:**

- Los nombres/descripciones de perfil son una síntesis interpretativa (sección 8 de
  `05_clustering`), no una categoría institucional oficial ni una verdad absoluta sobre las
  personas — la app debe mostrar este recordatorio de forma permanente (`lib.DISCLAIMER`).
- La app y sus datasets solo usan `IDPERSONA` (identificador ya anonimizado en el proyecto); no
  incorpora nombres reales. Un eventual despliegue institucional requeriría integrarse con los
  sistemas de RRHH bajo el control de acceso correspondiente — algo fuera del alcance de este
  prototipo.
- `SEXO` y `EDAD` se revisaron solo como chequeo de equidad (sección 4) y **no** se usan como
  filtro ni criterio dentro de la aplicación.
- La búsqueda semántica es opcional y tiene las limitaciones de la sección 5.